In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define schema configuration to use.
SCHEMA_CFG = 'configs/schema_1000p.yaml'
PHENOTYPES_CFG = 'configs/phenotypes.yaml'

# Paths
DATA_DIR = get_data_root_from_schema(schema_path=get_path(SCHEMA_CFG)) 
DATA_PROC_DIR = DATA_DIR / 'processed'
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Debug
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [12]:
# --- OPTION A: Load using pandas ---
from icare_risk.clinphen.io.loaders import load_from_pandas

# Load dataframes
dfs = load_from_pandas(
    schema_path=SCHEMA_CFG,
    domains_to_load=['episodes'],
    verbose=1
)

✅ Schema validation passed! All files exist and all physical columns match successfully.
Loaded... episodes (2.46MB) shape (10000, 11)


In [16]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
domains = ['episodes']

# Load
dfs = load_from_duckdb(
    schema_path=SCHEMA_CFG,
    phenotype_path=PHENOTYPES_CFG,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded... episodes (2.46MB) shape (10000, 11)
episodes: 10,000 rows -> 10,000 unique spells


In [17]:
# ------------------------------------------------
# Compute history of events (first, last, count)
# ------------------------------------------------
# Libraries
from icare_risk.clinphen.utils import compute_phenotype_history

# 1. Load the RAW events (not the aggregated wide matrix)
filename = '02a_derived_events_extracted_raw.parquet'
df_events_raw = pd.read_parquet(DATA_PROC_DIR / filename)

# 2. Include the index_admission column
df_episodes["index_admission"] = pd.to_datetime(
    df_episodes["ADMISSION_DATE"].astype(str) + " "
    + df_episodes["ADMISSION_TIME"].astype(str),
    errors="coerce",
)

# Print the occurrence counts of extracted events
event_counts = df_events_raw["event_name"].value_counts()
print(f"--- Extracted {len(event_counts)} Unique Event Components ---\n")
print(event_counts)

# 2. Compute recency for specific phenotypes (or leave None for all)
df = compute_phenotype_history(
    df_episodes=df_episodes,
    df_events_raw=df_events_raw,
    target_phenotypes=None,
    metrics=None,  # e.g., ["last", "first", "count"]
)

# 3. Save file
filename = '02b_derived_features_phenotypes.parquet'
df.to_parquet(DATA_PROC_DIR / filename)
print(f"\nFile successfully saved in: {DATA_DIR / filename}")

--- Extracted 33 Unique Event Components ---

event_name
charlson_hx_diabetes_uncomp__comp2          10081
kim_hx_recent_antibiotics_90d__comp0        10022
jones_hx_prior_antibiotics_30d__comp1       10022
kim_hx_recent_antibiotics_90d__comp1        10022
kim_hx_prior_antibiotics_90d__comp0         10022
jones_hx_prior_antibiotics_30d__comp0       10022
gavaghan_hx_prior_antibiotics_90d__comp1    10022
tumbarello_hx_recent_abx_90d__comp0         10022
charlson_hx_diabetes_uncomp__comp0            807
charlson_hx_renal_mod_sev__comp0              246
charlson_hx_stroke__comp0                     193
charlson_hx_pulmonary__comp0                  166
charlson_hx_dementia__comp0                   147
charlson_hx_chf__comp0                        103
charlson_hx_pud__comp0                         96
charlson_hx_mi__comp0                          90
charlson_hx_rheum__comp0                       77
tumbarello_hx_prior_esbl__comp0                60
kim_hx_prior_esbl__comp0                   

In [6]:
from icare_risk.clinphen.utils.debug import (
    print_phenotype_column_matrix,
    print_phenotype_columns_by_metric,
    summarize_phenotype_history
)

# Option A: See a compact per-phenotype matrix view
print_phenotype_column_matrix(df)

# Option B: See a flat, categorized list grouped by metric type
print_phenotype_columns_by_metric(df)

# Option C: Full distribution summary stats and report
df_report = summarize_phenotype_history(
    df, print_report=True, output_csv_path=None
)


--- Phenotype Column Matrix (30 phenotypes found) ---

  charlson_hx_aids                  | [✓] last   [✓] first   [✓] count
  charlson_hx_cancer_met            | [✓] last   [✓] first   [✓] count
  charlson_hx_cancer_solid          | [✓] last   [✓] first   [✓] count
  charlson_hx_chf                   | [✓] last   [✓] first   [✓] count
  charlson_hx_dementia              | [✓] last   [✓] first   [✓] count
  charlson_hx_diabetes_comp         | [✓] last   [✓] first   [✓] count
  charlson_hx_diabetes_uncomp       | [✓] last   [✓] first   [✓] count
  charlson_hx_hemiplegia            | [✓] last   [✓] first   [✓] count
  charlson_hx_hiv                   | [✓] last   [✓] first   [✓] count
  charlson_hx_leukemia              | [✓] last   [✓] first   [✓] count
  charlson_hx_liver_mild            | [✓] last   [✓] first   [✓] count
  charlson_hx_liver_mod_sev         | [✓] last   [✓] first   [✓] count
  charlson_hx_lymphoma              | [✓] last   [✓] first   [✓] count
  charlson_hx_mi     

In [18]:
print(df.columns.tolist())

['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'charlson_hx_mi_dslast', 'charlson_hx_mi_dsfirst', 'charlson_hx_mi_cnt', 'charlson_hx_chf_dslast', 'charlson_hx_chf_dsfirst', 'charlson_hx_chf_cnt', 'charlson_hx_pvd_dslast', 'charlson_hx_pvd_dsfirst', 'charlson_hx_pvd_cnt', 'charlson_hx_stroke_dslast', 'charlson_hx_stroke_dsfirst', 'charlson_hx_stroke_cnt', 'charlson_hx_dementia_dslast', 'charlson_hx_dementia_dsfirst', 'charlson_hx_dementia_cnt', 'charlson_hx_pulmonary_dslast', 'charlson_hx_pulmonary_dsfirst', 'charlson_hx_pulmonary_cnt', 'charlson_hx_rheum_dslast', 'charlson_hx_rheum_dsfirst', 'charlson_hx_rheum_cnt', 'charlson_hx_pud_dslast', 'charlson_hx_pud_dsfirst', 'charlson_hx_pud_cnt', 'charlson_hx_liver_mild_dslast', 'charlson_hx_liver_mild_dsfirst', 'charlson_hx_liver_mild_cnt',